# Steps 1 and 2
## Questions about the data

### Who assembled this data, and why?
Bonanza Creek LTER scientists in order to conduct their capture-recapture studies of snowshoe hares. 


### What does a single row in the data represent?
A capture event and the attributes of the hare that was captured. 

### When and where were the observations collected?
5 locales in the Tanana valley, from Tok in the east to Clear in the west from 1999 to 2002

### Does the dataset contain sensitive data?
No. If snowshoe hares were an endangered species than publishing  locations might be considered sensitive, but because theyre not I wouldnt consider there to be any sensitive data in this dataset. 

### What are the units for the hares’ weight?
Grams

### Does the metadata say how missing data is encoded?
No it does not. 

### Is there a publication associated with this dataset?
Yes, but I cannot find the original paper, its not available online.

## Citation
Kielland, K., F.S. Chapin, R.W. Ruess, and Bonanza Creek LTER. 2017. Snowshoe hare physical data in Bonanza Creek Experimental Forest: 1999-Present ver 22. Environmental Data Initiative. https://doi.org/10.6073/pasta/03dce4856d79b91557d8e6ce2cbcdc14

## Date of Access
Accessed 2026-10-17.

## Link to Archive
https://portal.edirepository.org/nis/mapbrowse?packageid=knb-lter-bnz.55.22

![Snowshoe Hare](https://thumb.wikimedia.org/wikipedia/commons/thumb/8/8a/SNOWSHOE_HARE_%28Lepus_americanus%29_%285-28-2015%29_quoddy_head%2C_washington_co%2C_maine_-01_%2818988734889%29.jpg/1920px-SNOWSHOE_HARE_%28Lepus_americanus%29_%285-28-2015%29_quoddy_head%2C_washington_co%2C_maine_-01_%2818988734889%29.jpg?utm_source=commons.wikimedia.org&utm_campaign=index&utm_content=thumbnail&_=20170313021652)

# 3. Data loading and preliminary exploration


In [23]:
# Load libraries in
import pandas as pd
import numpy as np 

# Store data in new dataframe hares
hares = pd.read_csv('55_Hare_Data_2012.txt')

In [24]:
# What are the dimensions of the data frame and what are the data types 
# of the columns? 
print(hares.shape)
print(hares.dtypes)

# Do the data types match what you would expect from each column?
hares.head()
# no they dont. b_key is supposed to be an integer yets its stored as a float
# similar thing with weight and hindft columns. 

(3380, 14)
date           object
time           object
grid           object
trap           object
l_ear          object
r_ear          object
sex            object
age            object
weight        float64
hindft        float64
notes          object
b_key         float64
session_id      int64
study          object
dtype: object


,date,time,grid,trap,l_ear,r_ear,sex,age,weight,hindft,notes,b_key,session_id,study
0,11/26/1998,NaN,bonrip,1A,414D096A08,NaN,NaN,NaN,1370.0,160.0,NaN,917.0,51,Population
1,11/26/1998,NaN,bonrip,2C,414D320671,NaN,M,NaN,1430.0,NaN,NaN,936.0,51,Population
2,11/26/1998,NaN,bonrip,2D,414D103E3A,NaN,M,NaN,1430.0,NaN,NaN,921.0,51,Population
3,11/26/1998,NaN,bonrip,2E,414D262D43,NaN,NaN,NaN,1490.0,135.0,NaN,931.0,51,Population
4,11/26/1998,NaN,bonrip,3B,414D2B4B58,NaN,NaN,NaN,1710.0,150.0,NaN,933.0,51,Population


In [25]:
# Are there any columns that have a significant number of NA values?
hares.isna().sum()
# Yes there are. time, age, notes, and hindft all have a singificant 
# amount of NA values (> 1000)

date             0
time          3116
grid             0
trap            12
l_ear           48
r_ear          169
sex            352
age           2111
weight         535
hindft        1747
notes         3137
b_key           47
session_id       0
study          163
dtype: int64

In [26]:
# What are the minimum and maximum values for the weight and hind feet 
# measurements?
print(hares[['weight', 'hindft']].min())
print(hares[['weight', 'hindft']].max())

weight     0.0
hindft    60.0
dtype: float64
weight    2365.0
hindft     160.0
dtype: float64


In [27]:
# What are the unique values for some of the categorical columns?
print(hares['sex'].sort_values().unique())
print(hares['grid'].sort_values().unique())
print(hares['age'].sort_values().unique())
print(hares['study'].sort_values().unique())

['?' 'F' 'F?' 'M' 'M?' 'f' 'f ' 'f?' 'm' 'm ' 'm?' 'pf' nan]
['Bonbs' 'bonbs' 'bonmat' 'bonmat ' 'bonrip']
['1' '1 yr' '1 yr.' '1.25' '1.5' '1/2/2013' '1/4/2013' '2 yrs' '2 yrs.'
 '2.25 yrs' '2.5 yrs' '3 yrs.' '3.25 yrs.' '3.5 yrs.' '3/4/2013' '?' 'A'
 'A 1.5' 'A 1/2' 'A 3/4' 'J' 'J 3/4' 'U' 'a' 'a 1 yr' 'a 1 yr.' 'a 2 yrs.'
 'a 3/4 yr.' 'j' 'u' nan]
['Collar' 'Metabolic' 'Metabolic/Collar' 'Population' nan]


# 4. Detecting messy values

### Allowed values for hare sex 
| Code | Definition | 
| ----------- | ----------- |
| m     | male       |
| f   | female       |
| m? | male not confirmed |

In [28]:
# Get the number of times each unique sex non-NA value appears.
hares['sex'].value_counts()

sex
F     1161
M      730
f      556
m      515
?       40
F?      10
f        4
m        4
f?       3
M?       2
m?       2
pf       1
Name: count, dtype: int64

In [29]:
# The purpose of dropna is to not count NA values. default it is 
# set to true. 
hares['sex'].value_counts(dropna=False)

sex
F      1161
M       730
f       556
m       515
NaN     352
?        40
F?       10
f         4
m         4
f?        3
M?        2
m?        2
pf        1
Name: count, dtype: int64

# 5. Brainstorm

### Write step-by-step instructions on how you would wrangle the hares data frame to clean the values in the sex column to have only two classes female and male.

1. Drop any values that contain question mark because we cant confidently ascertain as to whether it was a male or female. 
2. Drop na values, drop pf value
3. Remove whitespace
4. Uppercase all letters
5. From there we would rename F values to Female and M values to Male

In [38]:

#question_drop_mask = hares['sex'].str.contains('?', regex = False, na = False)
#cleaned_hares = hares[~question_drop_mask].copy()
# cleaned_hares['sex'].value_counts()
#cleaned_hares['sex_cleaned'] = cleaned_hares['sex'].dropna().str.strip().str.upper()
#cleaned_hares = cleaned_hares[cleaned_hares['sex_cleaned'] != 'PF']
#cleaned_hares['sex_cleaned'].value_counts()
#cleaned_hares.loc[cleaned_hares['sex_cleaned'] == 'M', 'sex_cleaned'] = 'Male'
#cleaned_hares.loc[cleaned_hares['sex_cleaned'] == 'F', 'sex_cleaned'] = 'Female'
#cleaned_hares['sex_cleaned'].value_counts()
cleaned_hares = hares.copy()
sex_clean = cleaned_hares['sex'].str.strip().str.upper()

cleaned_hares['sex_simple'] = np.select(
    [sex_clean == 'F', sex_clean == 'M'],   # conditions
    ['female', 'male'],                     # choices
    default= np.nan)                           # everything else

cleaned_hares['sex_simple'].value_counts(dropna=True)


sex_simple
female    1721
male      1249
nan        410
Name: count, dtype: int64